# TRELLIS.2 — RunPod GPU Setup & Inference

Tested on **A100 / H100 (24 GB+ VRAM)** with CUDA 12.4.  
Run cells top-to-bottom on a fresh RunPod PyTorch pod.

## 1. Verify GPU

In [1]:
!nvidia-smi

Mon Apr  6 16:41:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-32GB           On  |   00000000:86:00.0 Off |                    0 |
| N/A   33C    P0             42W /  300W |       0MiB /  32768MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Mar_28_02:18:24_PDT_2024
Cuda compilation tools, release 12.4, V12.4.131
Build cuda_12.4.r12.4/compiler.34097967_0


## 2. Clone Repo (skip if already present)

In [4]:
import os

REPO_DIR = "/workspace/TRELLIS.2"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/RahulBhalleySP/TRELLIS.2.git {REPO_DIR}
else:
    print(f"Repo already present at {REPO_DIR}")

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

# Persist repo path so bash cells can find it without hardcoding
with open('/tmp/trellis2_repo', 'w') as f:
    f.write(REPO_DIR)

Cloning into '/workspace/TRELLIS.2'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (212/212), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 376 (delta 66), reused 66 (delta 58), pack-reused 164 (from 1)
Receiving objects: 100% (376/376), 17.30 MiB | 17.92 MiB/s, done.
Resolving deltas: 100% (74/74), done.
Working directory: /workspace/TRELLIS.2


## 3. Install Dependencies

This mirrors `setup.sh`. Split into steps so you can re-run individual parts if one fails.

### 3a. PyTorch (CUDA 12.4)

In [5]:
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


### 3b. Basic Python packages

In [6]:
!pip install imageio imageio-ffmpeg tqdm easydict opencv-python-headless ninja \
             trimesh transformers gradio==6.0.1 tensorboard pandas lpips zstandard \
             kornia timm -q
!pip install git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8 -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [7]:
# Install system libs needed by pillow-simd (package names vary by distro/image)
!apt-get update -qq && apt-get install -y libjpeg-turbo8-dev zlib1g-dev -qq 2>/dev/null || \
 apt-get install -y libjpeg-dev zlib1g-dev -qq 2>/dev/null || \
 echo "apt install skipped (packages may already be present)"

# pillow-simd is a drop-in faster Pillow; fall back to stock Pillow if it fails
!pip install pillow-simd -q 2>/dev/null || pip install Pillow -q
!python -c "from PIL import Image; print('Pillow OK:', Image.__version__)"

Pillow OK: 9.5.0.post2


### 3c. Flash-Attention

In [8]:
!pip install flash-attn==2.7.3 -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


### 3d. CUDA extensions (nvdiffrast, nvdiffrec, CuMesh, FlexGEMM, O-Voxel)

Wheels are built once and cached to `/workspace/wheels/`.  
On subsequent runs the cached `.whl` files are installed directly — no recompilation.

In [ ]:
%%bash
export PIP_DISABLE_PIP_VERSION_CHECK=1
export PIP_ROOT_USER_ACTION=ignore
set -eo pipefail

WHEEL_DIR="/workspace/wheels"
WHEEL_ZIP="/workspace/wheels.zip"
MANIFEST="$WHEEL_DIR/manifest.txt"
EXT_DIR="/tmp/extensions"

# ── 1. Locate o-voxel source ──────────────────────────────────────────────────
# Exclude .git/ paths — git submodule storage lives at .git/modules/o-voxel
# and is a bare repo that cannot be used as a build source.
OVOXEL_SRC=$(find /workspace -maxdepth 4 -name "o-voxel" -type d \
             -not -path '*/.git/*' 2>/dev/null | head -1)

if [ -z "$OVOXEL_SRC" ]; then
    echo "[error] Could not find o-voxel/ under /workspace (excluding .git/)." >&2
    exit 1
fi
echo "[o-voxel src] $OVOXEL_SRC"

# Ensure Eigen submodule headers are present
if [ ! -f "$OVOXEL_SRC/third_party/eigen/Eigen/Dense" ]; then
    echo "[submodule] Initializing third_party/eigen ..."
    # Run from the parent repo root (one level up from o-voxel/)
    REPO_ROOT="$(dirname "$OVOXEL_SRC")"
    git -C "$REPO_ROOT" submodule update --init --recursive -- o-voxel/third_party/eigen -q \
        || git -C "$OVOXEL_SRC" submodule update --init --recursive -q
    echo "[submodule] Done."
fi

# ── 2. Bootstrap wheel cache ──────────────────────────────────────────────────
if [ -d "$WHEEL_DIR" ]; then
    echo "[cache] Using existing wheels/ folder."
elif [ -f "$WHEEL_ZIP" ]; then
    echo "[cache] Extracting wheels.zip ..."
    python3 -m zipfile -e "$WHEEL_ZIP" /workspace
    echo "[cache] Extraction done."
else
    echo "[cache] No cache found — building all wheels from source."
    mkdir -p "$WHEEL_DIR"
fi

touch "$MANIFEST"
mkdir -p "$EXT_DIR"

# ── Helpers ───────────────────────────────────────────────────────────────────

manifest_lookup() {
    local path
    path=$(grep "^${1}=" "$MANIFEST" 2>/dev/null | tail -1 | cut -d= -f2-)
    [ -f "$path" ] && echo "$path" || true
}

manifest_set() {
    sed -i "/^${1}=/d" "$MANIFEST" 2>/dev/null || true
    echo "${1}=${2}" >> "$MANIFEST"
}

new_wheels_since() {
    comm -13 \
        <(printf '%s\n' $1 | grep -v '^[[:space:]]*$' | sort -u) \
        <(printf '%s\n' $2 | grep -v '^[[:space:]]*$' | sort -u) \
    || true
}

# ── install_ext <pkg> <git-url|""> <branch|""> <src-dir> [force] ─────────────
install_ext() {
    local pkg="$1" repo="$2" branch="$3" src_dir="$4" force="${5:-}"

    echo ""
    echo "==> $pkg"

    if [ -z "$force" ]; then
        local cached
        cached=$(manifest_lookup "$pkg")
        if [ -n "$cached" ]; then
            echo "    [cache hit] $cached"
            pip install "$cached" --no-deps -q
            return 0
        fi
    else
        sed -i "/^${pkg}=/d" "$MANIFEST" 2>/dev/null || true
        # Remove stale wheel file so the before/after diff detects the rebuild.
        # pip wheel overwrites an existing file with the same name, making the
        # snapshot diff return empty. Deleting first guarantees a new entry appears.
        local pkg_norm
        pkg_norm=$(echo "$pkg" | tr '[:upper:]' '[:lower:]' | tr '-' '_')
        rm -f "$WHEEL_DIR"/${pkg_norm}*.whl "$WHEEL_DIR"/${pkg}*.whl 2>/dev/null || true
    fi

    if [ -n "$repo" ] && [ ! -d "$src_dir" ]; then
        local clone_args=(-q --recursive)
        [ -n "$branch" ] && clone_args+=(-b "$branch")
        git clone "${clone_args[@]}" "$repo" "$src_dir"
    fi

    if [ ! -d "$src_dir" ]; then
        echo "    [error] Source directory not found: $src_dir" >&2
        return 1
    fi

    echo "    Building wheel..."
    local before after new_whl
    before=$(ls "$WHEEL_DIR"/*.whl 2>/dev/null | sort || true)

    if ! pip wheel "$src_dir" --no-build-isolation --no-deps -w "$WHEEL_DIR" -q; then
        echo "    [error] pip wheel failed for $pkg" >&2
        return 1
    fi

    after=$(ls "$WHEEL_DIR"/*.whl 2>/dev/null | sort || true)
    new_whl=$(new_wheels_since "$before" "$after" | head -1)

    if [ -z "$new_whl" ]; then
        echo "    [error] No wheel produced for $pkg" >&2
        return 1
    fi

    pip install "$new_whl" --no-deps -q
    manifest_set "$pkg" "$new_whl"
    echo "    Cached: $new_whl"
}

# ── Build / install extensions ────────────────────────────────────────────────
install_ext "nvdiffrast" \
    "https://github.com/NVlabs/nvdiffrast.git" "v0.4.0" \
    "$EXT_DIR/nvdiffrast"

install_ext "nvdiffrec" \
    "https://github.com/JeffreyXiang/nvdiffrec.git" "renderutils" \
    "$EXT_DIR/nvdiffrec"

install_ext "cumesh" \
    "https://github.com/JeffreyXiang/CuMesh.git" "" \
    "$EXT_DIR/CuMesh"

install_ext "flex_gemm" \
    "https://github.com/JeffreyXiang/FlexGEMM.git" "" \
    "$EXT_DIR/FlexGEMM"

# o-voxel: sync latest (with submodules) and force-rebuild
rm -rf "$EXT_DIR/o-voxel"
cp -r "$OVOXEL_SRC" "$EXT_DIR/o-voxel"
install_ext "o_voxel" "" "" "$EXT_DIR/o-voxel" force

# ── Refresh zip (python3 -m zipfile: no system zip required) ─────────────────
echo ""
echo "Updating $WHEEL_ZIP ..."
python3 - <<'PYEOF'
import zipfile, pathlib

wheel_dir = pathlib.Path("/workspace/wheels")
zip_path  = pathlib.Path("/workspace/wheels.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(wheel_dir.rglob("*")):
        zf.write(f, f.relative_to(wheel_dir.parent))

print(f"Saved {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)")
PYEOF

echo ""
echo "Manifest:"
cat "$MANIFEST"
echo ""
echo "Wheels:"
ls -lh "$WHEEL_DIR"/*.whl 2>/dev/null | awk '{print $5, $9}' || echo "(none)"

## 4. Verify Imports

In [ ]:
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import sys
sys.path.insert(0, '/workspace/TRELLIS.2')

import cv2
import imageio
from PIL import Image
import o_voxel
from trellis2.pipelines import Trellis2ImageTo3DPipeline
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap

print("All imports OK")

## 5. Load Pipeline

Downloads ~4B-parameter model weights from HuggingFace on first run (~15 GB).

In [ ]:
pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.cuda()
print("Pipeline ready.")

## 6. Load Environment Map (for PBR rendering)

In [ ]:
envmap = EnvMap(
    torch.tensor(
        cv2.cvtColor(
            cv2.imread('assets/hdri/forest.exr', cv2.IMREAD_UNCHANGED),
            cv2.COLOR_BGR2RGB
        ),
        dtype=torch.float32,
        device='cuda'
    )
)
print("Environment map loaded.")

## 7. Pick Input Images

Any PNG / WebP with a clean background works.  
Change `image_paths` to your own files or upload them to `/workspace/`.

In [ ]:
import glob
from IPython.display import display

# Use a handful of the bundled example images
all_examples = sorted(glob.glob('assets/example_image/*.webp'))[:4]
all_examples += ['assets/example_image/T.png']  # also include the PNG

print("Images to process:")
for p in all_examples:
    img = Image.open(p).convert('RGB')
    img.thumbnail((128, 128))
    display(img)
    print(p)

## 8. Run Inference

Default pipeline type is `'1024_cascade'` (best quality).  
Use `'512'` for a quick test (~30 s vs ~3 min on A100).

In [ ]:
import time

PIPELINE_TYPE = '1024_cascade'  # '512' for fast test
OUTPUT_DIR = '/workspace/trellis2_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for img_path in all_examples:
    name = os.path.splitext(os.path.basename(img_path))[0]
    print(f"\n{'='*60}")
    print(f"Processing: {img_path}")

    image = Image.open(img_path)

    t0 = time.time()
    mesh = pipeline.run(image, pipeline_type=PIPELINE_TYPE)[0]
    mesh.simplify(16_777_216)  # nvdiffrast vertex limit
    print(f"Inference: {time.time()-t0:.1f}s | "
          f"vertices={mesh.vertices.shape[0]:,} faces={mesh.faces.shape[0]:,}")

    # --- Render turntable video ---
    t1 = time.time()
    frames = render_utils.make_pbr_vis_frames(
        render_utils.render_video(mesh, envmap=envmap)
    )
    video_path = os.path.join(OUTPUT_DIR, f'{name}.mp4')
    imageio.mimsave(video_path, frames, fps=15)
    print(f"Video saved: {video_path}  ({time.time()-t1:.1f}s)")

    # --- Export GLB ---
    t2 = time.time()
    glb = o_voxel.postprocess.to_glb(
        vertices          = mesh.vertices,
        faces             = mesh.faces,
        attr_volume       = mesh.attrs,
        coords            = mesh.coords,
        attr_layout       = mesh.layout,
        voxel_size        = mesh.voxel_size,
        aabb              = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        decimation_target = 1_000_000,
        texture_size      = 4096,
        remesh            = True,
        remesh_band       = 1,
        remesh_project    = 0,
        use_tqdm          = True,
    )
    glb_path = os.path.join(OUTPUT_DIR, f'{name}.glb')
    glb.export(glb_path, extension_webp=True)
    print(f"GLB saved:   {glb_path}  ({time.time()-t2:.1f}s)")

    # --- Export USDZ ---
    t3 = time.time()
    usdz_path = os.path.join(OUTPUT_DIR, f'{name}.usdz')
    o_voxel.postprocess.to_usdz(
        vertices          = mesh.vertices,
        faces             = mesh.faces,
        attr_volume       = mesh.attrs,
        coords            = mesh.coords,
        attr_layout       = mesh.layout,
        voxel_size        = mesh.voxel_size,
        aabb              = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        decimation_target = 1_000_000,
        texture_size      = 4096,
        remesh            = True,
        remesh_band       = 1,
        remesh_project    = 0,
        file_path         = usdz_path,
        use_tqdm          = True,
    )
    print(f"USDZ saved:  {usdz_path}  ({time.time()-t3:.1f}s)")

    torch.cuda.empty_cache()

print("\nAll done.")

## 9. Preview Outputs

In [ ]:
# List generated files
!ls -lh {OUTPUT_DIR}

In [ ]:
# Preview the first video inline
from IPython.display import Video
import glob

videos = sorted(glob.glob(f'{OUTPUT_DIR}/*.mp4'))
if videos:
    display(Video(videos[0], embed=True, width=512))

## 10. Download Outputs

Zip everything up for easy download from the RunPod file manager.

In [ ]:
!zip -r /workspace/trellis2_outputs.zip {OUTPUT_DIR}
print("Ready: /workspace/trellis2_outputs.zip")

---
## Optional: Single-Image Quick Test

Drop your own image at `/workspace/my_image.png` and run this cell.

In [ ]:
MY_IMAGE = '/workspace/my_image.png'  # <-- change this

if os.path.exists(MY_IMAGE):
    image = Image.open(MY_IMAGE)
    mesh = pipeline.run(image, pipeline_type='512')[0]  # '512' for speed
    mesh.simplify(16_777_216)

    glb = o_voxel.postprocess.to_glb(
        vertices=mesh.vertices, faces=mesh.faces,
        attr_volume=mesh.attrs, coords=mesh.coords,
        attr_layout=mesh.layout, voxel_size=mesh.voxel_size,
        aabb=[[-0.5,-0.5,-0.5],[0.5,0.5,0.5]],
        decimation_target=500_000, texture_size=2048,
        remesh=True, remesh_band=1, remesh_project=0,
    )
    out = '/workspace/my_output.glb'
    glb.export(out, extension_webp=True)
    print(f"Saved: {out}")

    torch.cuda.empty_cache()
else:
    print(f"File not found: {MY_IMAGE}")